# Fernando — S4 (#16) e S5 (#20)
Logística e árvore com validação estratificada por paciente, busca por AP e ajuste do limiar por F1 em validação independente. O teste final não participa das escolhas. Consulte [metodologia e resultados](../docs/sprints-4-5.md).

In [1]:
import sys, json
from pathlib import Path
import pandas as pd
from IPython.display import display
ROOT = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
OUTPUT = ROOT / 'reports/sprints-4-5'


## Reprodução opcional
Execute `python -m src.model_selection --jobs 2` na raiz para refazer os relatórios a partir do CSV. As células seguintes leem os resultados versionados, sem executar novamente a busca.

In [2]:
cv = pd.read_csv(OUTPUT / 's4_cv.csv')
display(cv)
display(cv.groupby('model')[['ap', 'roc_auc', 'precision', 'recall', 'f1']].agg(['mean', 'std']))

,model,fold,ap,roc_auc,precision,recall,f1
0,logistic,1,0.303695,0.661390,0.326389,0.016768,0.031897
1,logistic,2,0.306638,0.659190,0.377246,0.022484,0.042439
2,logistic,3,0.303680,0.659063,0.351852,0.020335,0.038449
3,logistic,4,0.307495,0.659335,0.417647,0.025330,0.047763
4,logistic,5,0.300528,0.653830,0.323171,0.018908,0.035726
5,decision_tree,1,0.229799,0.559998,0.301154,0.279344,0.289839
6,decision_tree,2,0.229397,0.557904,0.302698,0.276231,0.288860
7,decision_tree,3,0.224475,0.552379,0.288278,0.257938,0.272265
8,decision_tree,4,0.230242,0.559678,0.304313,0.274349,0.288555
9,decision_tree,5,0.231599,0.562302,0.306809,0.276489,0.290861


ap             roc_auc           precision            \
                   mean       std      mean       std      mean       std   
model                                                                       
decision_tree  0.229102  0.002716  0.558452  0.003738  0.300650  0.007226   
logistic       0.304407  0.002766  0.658561  0.002812  0.359261  0.039276   

                 recall                  f1            
                   mean       std      mean       std  
model                                                  
decision_tree  0.272870  0.008536  0.286076  0.007773  
logistic       0.020765  0.003294  0.039255  0.006119

In [3]:
results = json.loads((OUTPUT / 'results.json').read_text())
display(results['metadata'])
rows = []
for name in ['logistic', 'decision_tree']:
    result = results[name]
    print(name, result['best_params'], 'AP CV:', result['best_cv_ap'], 'limiar:', result['threshold'])
    for setting in ['test_default', 'test_adjusted']:
        rows.append({'model': name, 'setting': setting, **result[setting]})
display(pd.DataFrame(rows))

{'seed': 42,
 'folds': 5,
 'rows': 109889,
 'fit_rows': 70443,
 'validation_rows': 17277,
 'test_rows': 22169,
 'sklearn': '1.9.1',
 'numpy': '2.5.4',
 'pandas': '3.1.0',
 'features': ['age',
  'bolsa_familia',
  'hypertension',
  'diabetes',
  'alcoholism',
  'handicap',
  'waiting_days',
  'gender',
  'neighbourhood',
  'weekday'],
 'threshold_objective': 'F1 on independent validation',
 'search_objective': 'average_precision',
 'history_used': False}

logistic {'model__C': 0.01} AP CV: 0.30573135616599323 limiar: 0.19405641163216544
decision_tree {'model__max_depth': None, 'model__min_samples_leaf': 300} AP CV: 0.3409682788944698 limiar: 0.21288515406162464


,model,setting,accuracy,precision,recall,f1,roc_auc,pr_auc_ap,confusion_matrix,n_test,prevalence,contacts,captured_no_shows,precision_at_20,recall_at_20,random_expected_no_shows
0,logistic,test_default,0.793450,0.284211,0.012008,0.023042,0.648966,0.294753,"[[17536, 136], [4443, 54]]",22169,0.202851,4434,1484.0,0.334687,0.329998,899.44057
1,logistic,test_adjusted,0.641887,0.298194,0.565488,0.390480,0.648966,0.294753,"[[11687, 5985], [1954, 2543]]",22169,0.202851,4434,1484.0,0.334687,0.329998,899.44057
2,decision_tree,test_default,0.797149,0.000000,0.000000,0.000000,0.712774,0.334011,"[[17672, 0], [4497, 0]]",22169,0.202851,4434,1581.0,0.356563,0.351568,899.44057
3,decision_tree,test_adjusted,0.608327,0.306275,0.735824,0.432521,0.712774,0.334011,"[[10177, 7495], [1188, 3309]]",22169,0.202851,4434,1581.0,0.356563,0.351568,899.44057


## Limites
F1 é um compromisso técnico entre precisão e recall e não estima consultas recuperadas. O ranking de contato de 20% independe do limiar. A seleção do modelo final, SHAP, curvas e análise de negócio ficam com seus responsáveis. As features de histórico não são usadas neste experimento por paciente; mantemos o conjunto aprovado na S3.